# Ensemble Learning — Xinyi (6001CMD CW2)

**Work in progress.** Use Ben's shared train/test split and preprocessing, and Zyen's three tuned base models. This notebook must not invent or report test performance before the shared components are available.

Select two base models and the ensemble technique using training-only validation. Evaluate once on the held-out test set after the choice is final.

## 1. Load our helper code

In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / 'src' / 'ensemble.py').exists():
    root = root.parent
if not (root / 'src' / 'ensemble.py').exists():
    raise FileNotFoundError('Run from the repository root or notebooks folder')
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_val_score
from src.ensemble import choose_two_models, make_ensembles, evaluate_candidates
print('Ensemble helper imported successfully')

## 2. Shared interface — waiting for Ben and Zyen

Ben must provide the agreed `X_train`, `X_test`, `y_train`, `y_test` split. Zyen must supply exactly three tuned, cloneable scikit-learn estimators, preferably **Pipelines** that include their own training-only preprocessing. For soft voting both selected models need `predict_proba`.

The target is encoded 1 = yes, 0 = no. Do **not** fit transformations on the full dataset or modify Ben/Zyen's notebooks.

In [ ]:
# Replace these placeholders ONLY when the actual shared interfaces are available.
# Example (illustrative only, not real module names):
# from src.shared_preprocessing import get_split
# from src.base_models import get_tuned_models
# X_train, X_test, y_train, y_test = get_split(...)
# base_models = get_tuned_models(...)
X_train = X_test = y_train = y_test = None
base_models = {}
RUN_EXPERIMENTS = False
print('Waiting for Ben and Zyen. Experiment enabled?', RUN_EXPERIMENTS)

## 3. Select two models using training-fold validation

Use F1 for the positive class as the primary criterion because the Bank Marketing classes are imbalanced; also inspect recall, precision, ROC-AUC and prediction diversity. Document why the third model is excluded.

No final test data are consulted during selection.

In [ ]:
if RUN_EXPERIMENTS:
    assert all(x is not None for x in [X_train, X_test, y_train, y_test])
    assert len(base_models) == 3, 'Zyen must provide exactly three models'
    assert set(np.unique(y_train)) == {0, 1}
    outer_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    cv_f1 = {}
    for name, model in base_models.items():
        scores = cross_val_score(clone(model), X_train, y_train, scoring='f1', cv=outer_cv)
        cv_f1[name] = float(scores.mean())
        print(name, 'train-CV F1:', round(cv_f1[name], 4))
    selected = choose_two_models(cv_f1)
    print('Candidate pair:', selected, '| Excluded:', set(base_models)-set(selected))
else:
    print('Not run: waiting for shared models and data split.')

## 4. Compare ensemble methods on TRAINING folds only

The final ensemble method will be selected by validation performance, not by looking at the final test set. Stacking trains its meta-model using internal cross-validation.

In [ ]:
if RUN_EXPERIMENTS:
    candidate_ensembles = make_ensembles(base_models, selected)
    method_f1 = {}
    for name, model in candidate_ensembles.items():
        scores = cross_val_score(clone(model), X_train, y_train, scoring='f1', cv=outer_cv)
        method_f1[name] = float(scores.mean())
        print(name, 'train-CV F1:', round(method_f1[name], 4))
    final_method = max(method_f1, key=method_f1.get)
    print('Validation-selected method:', final_method)

## 5. Final evaluation — exactly 3 base models + 1 ensemble

Run only when model selection and tuning have already been finalized. Avoid repeatedly checking the held-out test set. Report Accuracy, Precision, Recall, F1, ROC-AUC, confusion matrices, runtime, and discuss predictive performance versus deployment costs.

In [ ]:
if RUN_EXPERIMENTS:
    results, fitted_models, confusion_matrices = evaluate_candidates(
        base_models, selected, X_train, y_train, X_test, y_test,
        ensemble_method=final_method,
    )
    display(results.round(4))
    for name, cm in confusion_matrices.items():
        print(name, '\n', cm)
else:
    print('Final test set untouched; no performance results yet.')

## 6. Visual evaluation — model comparison, confusion matrices and ROC curves

These figures must use **actual held-out test results** after Ben's agreed split and Zyen's three tuned models are integrated. With `RUN_EXPERIMENTS = False`, no models will train and no simulated scores will appear. Choose the selected ensemble method using training cross-validation, not these test figures. Save or export figures only after evaluating your final locked model choices.

In [ ]:
from src.evaluation_plots import (
    plot_model_comparison,
    plot_confusion_matrices,
    plot_roc_curves,
)

if RUN_EXPERIMENTS:
    # Run only after the final ensemble has been selected using training CV.
    import matplotlib.pyplot as plt

    comparison_fig = plot_model_comparison(results)
    plt.show()

    confusion_fig = plot_confusion_matrices(confusion_matrices)
    plt.show()

    roc_fig = plot_roc_curves(fitted_models, X_test, y_test)
    plt.show()
else:
    print("Visualizations ready. Waiting for real model results; no plots generated.")

## 7. Report and viva notes

After running real experiments, write in your own words:

1. Why the two base models were selected and why the third was excluded (with training-CV evidence).
2. How Soft Voting or Stacking works and why the chosen strategy fits the task.
3. The *actual* final-test metric table comparing three base models with one ensemble.
4. False-negative versus false-positive implications for marketing, class imbalance and overfitting.
5. Training cost, interpretability and evidence-based deployment recommendation — **even if a base model is better than the ensemble**.

Follow the course AI-use disclosure requirements. Keep `bank-full.csv` out of GitHub.